In [27]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch import Tensor
import matplotlib.pyplot as plt
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import random_split

In [28]:
# 1. 사용할 디바이스 지정 (NVIDIA GPU -> Mac GPU -> CPU 순서)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # M1/M2/M3 맥북 유저용
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cuda


### 학습 데이터 설정


In [29]:
train_dataset = datasets.FashionMNIST(
    root="data/fashion_mnist_data/", train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = datasets.FashionMNIST(
    root="data/fashion_mnist_data/", train=False, transform=transforms.ToTensor(), download=True
)

In [30]:
train_dataset_size = int(len(train_dataset) * 0.85)
validation_dataset_size = len(train_dataset) - train_dataset_size

train_dataset, validation_dataset = random_split(
    train_dataset, [train_dataset_size, validation_dataset_size]
)

print(len(train_dataset), len(validation_dataset), len(test_dataset))

51000 9000 10000


In [31]:
BATCH_SIZE = 64

train_dataset_loader = DataLoader(
    dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
validation_dataset_loader = DataLoader(
    dataset=validation_dataset, batch_size=BATCH_SIZE, shuffle=True
)
test_dataset_loader = DataLoader(
    dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

### 모델 생성


In [32]:
class MlpModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 10),
        )

    def forward(self, data):
        logits = self.layer_stack(data)
        return logits

In [33]:
model = MlpModel()

loss_function = (
    nn.CrossEntropyLoss()
)  # CrossEntropyLoss 손실함수에는 softmax 함수 포함되어 있음
optim = torch.optim.Adam(model.parameters(), lr=1e-3)

In [34]:
from torchinfo import summary

summary(model, input_size=(1, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
MlpModel                                 [1, 10]                   --
├─Sequential: 1-1                        [1, 10]                   --
│    └─Flatten: 2-1                      [1, 784]                  --
│    └─Linear: 2-2                       [1, 1024]                 803,840
│    └─ReLU: 2-3                         [1, 1024]                 --
│    └─Dropout: 2-4                      [1, 1024]                 --
│    └─Linear: 2-5                       [1, 512]                  524,800
│    └─ReLU: 2-6                         [1, 512]                  --
│    └─Dropout: 2-7                      [1, 512]                  --
│    └─Linear: 2-8                       [1, 10]                   5,130
Total params: 1,333,770
Trainable params: 1,333,770
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 1.33
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 5.34
Estimated Tot

### 학습 및 검증 함수 선언


In [35]:
def model_train(
    dataloader: DataLoader,
    model: MlpModel,
    loss_function: nn.CrossEntropyLoss,
    optim: torch.optim.SGD,
):

    model.train()

    loss_sum = train_correct = total_count = 0
    batch_count = len(dataloader)

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = loss_function(outputs, labels)

        optim.zero_grad()
        loss.backward()
        optim.step()

        loss_sum += loss.item()

        total_count += labels.size(0)
        train_correct += ((torch.argmax(outputs, 1) == labels)).sum().item()

    avg_loss = loss_sum / batch_count
    avg_accuracy = 100 * train_correct / total_count

    return (avg_loss, avg_accuracy)

In [36]:
def model_evaluate(
    dataloader: DataLoader,
    model: MlpModel,
    loss_function: nn.CrossEntropyLoss,
):

    model.eval()

    with torch.no_grad():

        loss_sum = correct_list = total_count = 0
        batch_count = len(dataloader)

        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            loss_sum += loss.item()

            total_count += labels.size(0)
            correct_list += ((torch.argmax(outputs, 1) == labels)).sum().item()

        avg_loss = loss_sum / batch_count
        avg_accuracy = 100 * correct_list / total_count

    return (avg_loss, avg_accuracy)

In [37]:
def model_test(
    dataloader: DataLoader,
    model: MlpModel,
    loss_function: nn.CrossEntropyLoss,
):

    model.eval()

    with torch.no_grad():

        loss_sum = correct_list = total_count = 0
        batch_count = len(dataloader)

        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            loss_sum += loss.item()

            total_count += labels.size(0)
            correct_list += ((torch.argmax(outputs, 1) == labels)).sum().item()

        avg_loss = loss_sum / batch_count
        avg_accuracy = 100 * correct_list / total_count

    print("accuracy:", avg_accuracy)
    print("loss:", avg_loss)

In [38]:
train_loss_list = []
train_accuracy_list = []

val_loss_list = []
val_accuracy_list = []

EPOCHS = 20

for epoch in range(EPOCHS):

    # ================== model train ======================
    train_avg_loss, train_avg_accuracy = model_train(
        train_dataset_loader, model, loss_function, optim
    )

    train_loss_list.append(train_avg_loss)
    train_accuracy_list.append(train_avg_accuracy)
    # =====================================================

    # ================== model evaluation =================
    val_avg_loss, val_avg_accuracy = model_evaluate(
        validation_dataset_loader, model, loss_function
    )

    val_loss_list.append(val_avg_loss)
    val_accuracy_list.append(val_avg_accuracy)
    # =====================================================

    print(
        f"epoch: {epoch}, train loss= {train_avg_loss}, train accuracy= {train_avg_accuracy},"
        f"validation loss= {val_avg_loss}, validation accuracy= {val_avg_accuracy}"
    )

epoch: 0, train loss= 0.5355477463383896, train accuracy= 80.47647058823529,validation loss= 0.41279969020938195, validation accuracy= 84.95555555555555
epoch: 1, train loss= 0.40918607834067516, train accuracy= 85.04117647058824,validation loss= 0.3811403637659465, validation accuracy= 86.2
epoch: 2, train loss= 0.37396736463608976, train accuracy= 86.24901960784314,validation loss= 0.36314444349589925, validation accuracy= 86.73333333333333
epoch: 3, train loss= 0.3542522390558848, train accuracy= 86.91176470588235,validation loss= 0.36278338001129473, validation accuracy= 86.38888888888889
epoch: 4, train loss= 0.3376943114747666, train accuracy= 87.47450980392156,validation loss= 0.35597268497267515, validation accuracy= 86.92222222222222
epoch: 5, train loss= 0.32384094634237076, train accuracy= 88.00588235294117,validation loss= 0.3463197334649715, validation accuracy= 86.94444444444444
epoch: 6, train loss= 0.31216453543047384, train accuracy= 88.24901960784314,validation loss= 

### 모델 테스트

In [39]:
model_test(test_dataset_loader, model, loss_function)

accuracy: 89.37
loss: 0.3121356944179839
